In [1]:
import gbd_mapping
import risk_distributions
import pathlib
import pandas as pd, numpy as np
import vivarium_inputs
from vivarium_inputs import utility_data, globals as vi_globals, utilities as vi_utils
from vivarium_gbd_access import gbd
import os, contextlib, warnings, loguru

from vivarium_inputs.validation.raw import DataDoesNotExistError, DataAbnormalError
from tqdm.notebook import tqdm

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
pd.set_option("display.max_columns", 30)

In [3]:
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [4]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [5]:
# Parameters
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"


In [6]:
index_cols = ["sex", "age_start", "age_end", "wealth_quintile"]

age_group_ids = [
    2,3,
    388,389,
    6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235
]
sex_ids = [1,2]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [7]:
fortification_hemoglobin_mean_difference = (
    pd.read_csv('../0100_data_prep/results/iron/fortification_hemoglobin_effects.csv')
        .set_index('vehicle_name').value
        .loc[vehicle]
)
fortification_hemoglobin_mean_difference

3.25

In [8]:
effective_baseline_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/baseline_fortification/effective_coverage/{location}.csv')
)
assert (effective_baseline_coverage.vehicle_name == vehicle).all()
effective_baseline_coverage = effective_baseline_coverage.drop(columns=["vehicle_name"])
effective_baseline_coverage

,wealth_quintile,sex,age_start,age_end,value
0,fourth,Female,0,5,0.239271
1,fourth,Female,5,15,0.298223
2,fourth,Female,15,30,0.278779
3,fourth,Female,30,50,0.282996
4,fourth,Female,50,125,0.282733
5,fourth,Male,0,5,0.239590
6,fourth,Male,5,15,0.298466
7,fourth,Male,15,30,0.281030
8,fourth,Male,30,50,0.275204
9,fourth,Male,50,125,0.280098


In [9]:
def expand(df):
    for col in sorted(list(set(df.columns) - {'value'})):
        if df[col].isnull().any():
            df = pd.concat([
                df[df[col].notnull()],
                *[df[df[col].isnull()].assign(**{col: value}) for value in df[df[col].notnull()][col].unique()]
            ])
    
    return df

In [10]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_baseline_coverage.columns:
        effective_baseline_coverage[col] = fill_value
    else:
        effective_baseline_coverage[col] = effective_baseline_coverage[col].fillna(fill_value)

In [11]:
effective_baseline_coverage = expand(effective_baseline_coverage)
effective_baseline_coverage

,wealth_quintile,sex,age_start,age_end,value
0,fourth,Female,0,5,0.239271
1,fourth,Female,5,15,0.298223
2,fourth,Female,15,30,0.278779
3,fourth,Female,30,50,0.282996
4,fourth,Female,50,125,0.282733
5,fourth,Male,0,5,0.239590
6,fourth,Male,5,15,0.298466
7,fourth,Male,15,30,0.281030
8,fourth,Male,30,50,0.275204
9,fourth,Male,50,125,0.280098


In [12]:
effective_counterfactual_coverage = (
    pd.read_csv(f'../0100_data_prep/results/iron/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv')
)
assert (effective_counterfactual_coverage.vehicle_name == vehicle).all()
effective_counterfactual_coverage = effective_counterfactual_coverage.drop(columns=["vehicle_name"])
effective_counterfactual_coverage

,sex,age_start,age_end,wealth_quintile,value
0,Female,0,5,lowest,0.480408
1,Female,0,5,second,0.489627
2,Female,0,5,middle,0.485258
3,Female,0,5,fourth,0.465105
4,Female,0,5,highest,0.414790
5,Female,5,15,lowest,0.518085
6,Female,5,15,second,0.516195
7,Female,5,15,middle,0.503735
8,Female,5,15,fourth,0.480685
9,Female,5,15,highest,0.422331


In [13]:
for col, fill_value in [("age_start", 0), ("age_end", 125)]:
    if col not in effective_counterfactual_coverage.columns:
        effective_counterfactual_coverage[col] = fill_value
    else:
        effective_counterfactual_coverage[col] = effective_counterfactual_coverage[col].fillna(fill_value)

In [14]:
effective_counterfactual_coverage = expand(effective_counterfactual_coverage)
effective_counterfactual_coverage

,sex,age_start,age_end,wealth_quintile,value
0,Female,0,5,lowest,0.480408
1,Female,0,5,second,0.489627
2,Female,0,5,middle,0.485258
3,Female,0,5,fourth,0.465105
4,Female,0,5,highest,0.414790
5,Female,5,15,lowest,0.518085
6,Female,5,15,second,0.516195
7,Female,5,15,middle,0.503735
8,Female,5,15,fourth,0.480685
9,Female,5,15,highest,0.422331


In [15]:
non_pregnant_pop = (
    pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv')
).pipe(lambda df: df[df.pregnant == "not_pregnant"]).drop(columns="pregnant")
non_pregnant_pop = non_pregnant_pop.set_index([c for c in non_pregnant_pop.columns if c != 'value']).value
non_pregnant_pop

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             49649.700497
                               second             43575.622144
                               middle             38689.356750
                               fourth             36440.833935
                               highest            29202.585338
                                                      ...     
Male    95.0       125.000000  lowest             15176.692284
                               second             15848.509818
                               middle             16370.176208
                               fourth             17265.313670
                               highest            21289.473456
Name: value, Length: 250, dtype: float64

In [16]:
population_age_groups = non_pregnant_pop.reset_index()[["age_start", "age_end"]].drop_duplicates().sort_values("age_start")
population_age_groups

,age_start,age_end
0,0.000000,0.019178
5,0.019178,0.076712
10,0.076712,0.500000
15,0.500000,1.000000
20,1.000000,2.000000
25,2.000000,5.000000
30,5.000000,10.000000
35,10.000000,15.000000
40,15.000000,20.000000
45,20.000000,25.000000


In [17]:
def map_to_population_age_groups(df):
    return (
        population_age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )

In [18]:
effective_baseline_coverage = map_to_population_age_groups(effective_baseline_coverage).set_index([c for c in effective_baseline_coverage.columns if c != 'value']).value
effective_counterfactual_coverage = map_to_population_age_groups(effective_counterfactual_coverage).set_index([c for c in effective_counterfactual_coverage.columns if c != 'value']).value

In [19]:
delta_effective_coverage = effective_counterfactual_coverage.sub(effective_baseline_coverage)
delta_effective_coverage

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    fourth             0.225834
                               highest            0.277436
                               lowest             0.185658
                               middle             0.206281
                               second             0.190897
                                                    ...   
Male    95.0       125.000000  fourth             0.187922
                               highest            0.263533
                               lowest             0.129487
                               middle             0.163211
                               second             0.140935
Name: value, Length: 250, dtype: float64

In [20]:
def reshape_to_vivarium_format(df, location):
    df = vi_utils.reshape(df, value_cols=[c for c in df.columns if 'draw_' in c])
    df = vi_utils.scrub_gbd_conventions(df, location)
    df = vi_utils.split_interval(df, interval_column="age", split_column_prefix="age")
    df = vi_utils.split_interval(df, interval_column="year", split_column_prefix="year")
    df = vi_utils.sort_hierarchical_data(df)
    df.index = df.index.droplevel("location")
    return df

In [21]:
me_ids = {
    "hemoglobin_mean": 10487,
    "hemoglobin_sd": 10488,
}

In [22]:
location_id = utility_data.get_location_id(location.title())
hgb_mean = gbd.get_modelable_entity_draws(me_id=me_ids["hemoglobin_mean"], location_id=location_id, year_id=2021)
hgb_mean = reshape_to_vivarium_format(hgb_mean, location.title()).droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS].copy()
hgb_mean

draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    141.027303  141.174892  143.047093  142.497768   
       0.019178  0.076712    128.185915  129.096397  129.130806  127.546092   
       0.076712  0.500000    103.541148  104.218523  102.321094  104.327579   
       0.500000  1.000000    101.917809   99.823676  100.483745  100.428266   
       1.000000  2.000000    101.808200  103.457343  102.146844  103.058795   
       2.000000  5.000000    106.297867  106.219133  105.789212  107.260720   
       5.000000  10.000000   112.932920  111.768831  112.598540  114.147894   
       10.000000 15.000000   115.889117  117.854010  116.651884  116.435834   
       15.000000 20.000000   117.312802  117.869717  117.696658  118.290512   
       20.000000 25.000000   118.617583  117.381244  117.562883  116.463839   
       25.000000 30.000000   118.136225  118.984193  118.330466  117.484197   
       30.000000 35.000000   118.468947  117.409679  117.886880  119.228899   
       35.000000 40.000000   117.893922  117.756202  118.469227  118.035053   
       40.000000 45.000000   117.705099  117.438677  117.891058  117.526008   
       45.000000 50.000000   118.863416  118.956356  117.862423  118.694166   
       50.000000 55.000000   117.380298  115.684488  117.082839  117.521103   
       55.000000 60.000000   115.610201  115.858632  116.824778  116.971145   
       60.000000 65.000000   115.852172  115.273742  114.679645  115.057166   
       65.000000 70.000000   112.799956  114.029249  112.513289  113.319053   
       70.000000 75.000000   111.763028  110.907571  111.248801  110.229346   
       75.000000 80.000000   106.694860  105.688161  106.061586  105.665603   
       80.000000 85.000000   105.375787  105.408475  105.314517  107.634883   
       85.000000 90.000000   100.573661  100.988738   98.813280   99.706145   
       90.000000 95.000000    97.373246   97.761692   97.824092   97.656022   
       95.000000 125.000000   95.150708   95.688588   94.653440   94.380913   
Male   0.000000  0.019178    146.600654  149.103908  146.074897  145.748027   
       0.019178  0.076712    128.593451  128.987080  131.564522  129.046613   
       0.076712  0.500000    102.882001  103.484281  102.379468  102.302975   
       0.500000  1.000000     98.720550  100.044084   99.267918  100.076804   
       1.000000  2.000000    101.799264  101.036897  102.067002  102.340244   
       2.000000  5.000000    107.195733  107.108687  106.303427  107.176671   
       5.000000  10.000000   115.943763  116.573107  114.252198  113.966774   
       10.000000 15.000000   125.861685  126.103005  123.731518  125.853467   
       15.000000 20.000000   138.148186  139.196158  138.577766  140.412738   
       20.000000 25.000000   145.705670  143.628038  144.447657  145.036625   
       25.000000 30.000000   143.206791  144.192168  144.708908  142.243978   
       30.000000 35.000000   143.104472  145.407658  144.512082  143.314688   
       35.000000 40.000000   143.022415  141.891966  143.895938  141.822267   
       40.000000 45.000000   142.575032  141.418982  142.295188  141.503041   
       45.000000 50.000000   141.450799  142.687529  139.176123  140.952480   
       50.000000 55.000000   138.187591  140.164931  139.259715  141.275545   
       55.000000 60.000000   135.036605  134.464782  134.025679  133.016133   
       60.000000 65.000000   131.931505  132.888573  130.701815  131.483537   
       65.000000 70.000000   128.500505  127.922563  128.410141  128.344826   
       70.000000 75.000000   122.092349  122.015506  121.558645  122.075910   
       75.000000 80.000000   113.648012  114.150699  114.133169  113.393346   
       80.000000 85.000000   111.213129  111.511647  111.224695  113.696862   
       85.000000 90.000000   108.957315  109.468162  107.835589  108.453981   
       90.000000 95.000000   104.826207  102.838567  103.076523  105.765753   
    

In [23]:
hemoglobin_mean_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv')
hemoglobin_mean_disparities = (
    map_to_population_age_groups(hemoglobin_mean_disparities[hemoglobin_mean_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_mean_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             103.655561
                               second             105.257986
                               middle             105.397530
                               fourth             106.765336
                               highest            107.989017
                                                     ...    
Male    95.0       125.000000  lowest             115.022635
                               second             116.108207
                               middle             116.488978
                               fourth             117.310740
                               highest            118.402649
Name: value, Length: 250, dtype: float64

In [24]:
wealth_quintile_probabilities = pd.read_csv(f'../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv')
wealth_quintile_probabilities

,sex,age_start,age_end,pregnant,lowest,second,middle,fourth,highest
0,Female,0.0,5.0,not_pregnant,0.251317,0.220571,0.195838,0.184456,0.147818
1,Female,5.0,15.0,not_pregnant,0.269038,0.219459,0.193766,0.172569,0.145167
2,Female,15.0,30.0,not_pregnant,0.177853,0.202279,0.211418,0.210441,0.198009
3,Female,15.0,30.0,pregnant,0.221117,0.222530,0.215087,0.182296,0.158970
4,Female,30.0,50.0,not_pregnant,0.172597,0.187138,0.199031,0.213495,0.227738
5,Female,30.0,50.0,pregnant,0.316131,0.181401,0.129805,0.169147,0.203517
6,Female,50.0,125.0,not_pregnant,0.188708,0.187540,0.191253,0.199636,0.232863
7,Male,0.0,5.0,not_pregnant,0.242600,0.215610,0.200229,0.183923,0.157639
8,Male,5.0,15.0,not_pregnant,0.261604,0.217675,0.190718,0.176545,0.153458
9,Male,15.0,30.0,not_pregnant,0.169172,0.205794,0.214029,0.208330,0.202675


In [25]:
wealth_quintile_probabilities = (
    map_to_population_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end"])
)
wealth_quintile_probabilities.columns.name = 'wealth_quintile'
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             0.251317
                               second             0.220571
                               middle             0.195838
                               fourth             0.184456
                               highest            0.147818
                                                    ...   
Male    95.0       125.000000  lowest             0.176575
                               second             0.184392
                               middle             0.190461
                               fourth             0.200876
                               highest            0.247696
Length: 250, dtype: float64

In [26]:
assert np.allclose(wealth_quintile_probabilities.groupby(["sex", "age_start", "age_end"]).sum(), 1.0)

In [27]:
pre_disparity_groups = hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
pre_disparity_groups

draw_0      draw_1      draw_2      draw_3  \
sex    age_start age_end                                                      
Female 0.000000  0.019178    141.027303  141.174892  143.047093  142.497768   
       0.019178  0.076712    128.185915  129.096397  129.130806  127.546092   
       0.076712  0.500000    103.541148  104.218523  102.321094  104.327579   
       0.500000  1.000000    101.917809   99.823676  100.483745  100.428266   
       1.000000  2.000000    101.808200  103.457343  102.146844  103.058795   
       2.000000  5.000000    106.297867  106.219133  105.789212  107.260720   
       5.000000  10.000000   112.932920  111.768831  112.598540  114.147894   
       10.000000 15.000000   115.889117  117.854010  116.651884  116.435834   
       15.000000 20.000000   117.312802  117.869717  117.696658  118.290512   
       20.000000 25.000000   118.617583  117.381244  117.562883  116.463839   
       25.000000 30.000000   118.136225  118.984193  118.330466  117.484197   
       30.000000 35.000000   118.468947  117.409679  117.886880  119.228899   
       35.000000 40.000000   117.893922  117.756202  118.469227  118.035053   
       40.000000 45.000000   117.705099  117.438677  117.891058  117.526008   
       45.000000 50.000000   118.863416  118.956356  117.862423  118.694166   
       50.000000 55.000000   117.380298  115.684488  117.082839  117.521103   
       55.000000 60.000000   115.610201  115.858632  116.824778  116.971145   
       60.000000 65.000000   115.852172  115.273742  114.679645  115.057166   
       65.000000 70.000000   112.799956  114.029249  112.513289  113.319053   
       70.000000 75.000000   111.763028  110.907571  111.248801  110.229346   
       75.000000 80.000000   106.694860  105.688161  106.061586  105.665603   
       80.000000 85.000000   105.375787  105.408475  105.314517  107.634883   
       85.000000 90.000000   100.573661  100.988738   98.813280   99.706145   
       90.000000 95.000000    97.373246   97.761692   97.824092   97.656022   
       95.000000 125.000000   95.150708   95.688588   94.653440   94.380913   
Male   0.000000  0.019178    146.600654  149.103908  146.074897  145.748027   
       0.019178  0.076712    128.593451  128.987080  131.564522  129.046613   
       0.076712  0.500000    102.882001  103.484281  102.379468  102.302975   
       0.500000  1.000000     98.720550  100.044084   99.267918  100.076804   
       1.000000  2.000000    101.799264  101.036897  102.067002  102.340244   
       2.000000  5.000000    107.195733  107.108687  106.303427  107.176671   
       5.000000  10.000000   115.943763  116.573107  114.252198  113.966774   
       10.000000 15.000000   125.861685  126.103005  123.731518  125.853467   
       15.000000 20.000000   138.148186  139.196158  138.577766  140.412738   
       20.000000 25.000000   145.705670  143.628038  144.447657  145.036625   
       25.000000 30.000000   143.206791  144.192168  144.708908  142.243978   
       30.000000 35.000000   143.104472  145.407658  144.512082  143.314688   
       35.000000 40.000000   143.022415  141.891966  143.895938  141.822267   
       40.000000 45.000000   142.575032  141.418982  142.295188  141.503041   
       45.000000 50.000000   141.450799  142.687529  139.176123  140.952480   
       50.000000 55.000000   138.187591  140.164931  139.259715  141.275545   
       55.000000 60.000000   135.036605  134.464782  134.025679  133.016133   
       60.000000 65.000000   131.931505  132.888573  130.701815  131.483537   
       65.000000 70.000000   128.500505  127.922563  128.410141  128.344826   
       70.000000 75.000000   122.092349  122.015506  121.558645  122.075910   
       75.000000 80.000000   113.648012  114.150699  114.133169  113.393346   
       80.000000 85.000000   111.213129  111.511647  111.224695  113.696862   
       85.000000 90.000000   108.957315  109.468162  107.835589  108.453981   
       90.000000 95.000000   104.826207  102.838567  103.076523  105.765753   
    

In [28]:
hgb_mean = hgb_mean.mul(hemoglobin_mean_disparities, axis=0)

In [29]:
scale_factor = pre_disparity_groups / hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
scale_factor

draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.009473  0.009473  0.009473  0.009473  0.009473   
       0.019178  0.076712    0.009473  0.009473  0.009473  0.009473  0.009473   
       0.076712  0.500000    0.009473  0.009473  0.009473  0.009473  0.009473   
       0.500000  1.000000    0.009473  0.009473  0.009473  0.009473  0.009473   
       1.000000  2.000000    0.009473  0.009473  0.009473  0.009473  0.009473   
       2.000000  5.000000    0.009473  0.009473  0.009473  0.009473  0.009473   
       5.000000  10.000000   0.009011  0.009011  0.009011  0.009011  0.009011   
       10.000000 15.000000   0.009011  0.009011  0.009011  0.009011  0.009011   
       15.000000 20.000000   0.008569  0.008569  0.008569  0.008569  0.008569   
       20.000000 25.000000   0.008569  0.008569  0.008569  0.008569  0.008569   
       25.000000 30.000000   0.008569  0.008569  0.008569  0.008569  0.008569   
       30.000000 35.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       35.000000 40.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       40.000000 45.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       45.000000 50.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       50.000000 55.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       55.000000 60.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       60.000000 65.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       65.000000 70.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       70.000000 75.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       75.000000 80.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       80.000000 85.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       85.000000 90.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       90.000000 95.000000   0.008565  0.008565  0.008565  0.008565  0.008565   
       95.000000 125.000000  0.008565  0.008565  0.008565  0.008565  0.008565   
Male   0.000000  0.019178    0.009480  0.009480  0.009480  0.009480  0.009480   
       0.019178  0.076712    0.009480  0.009480  0.009480  0.009480  0.009480   
       0.076712  0.500000    0.009480  0.009480  0.009480  0.009480  0.009480   
       0.500000  1.000000    0.009480  0.009480  0.009480  0.009480  0.009480   
       1.000000  2.000000    0.009480  0.009480  0.009480  0.009480  0.009480   
       2.000000  5.000000    0.009480  0.009480  0.009480  0.009480  0.009480   
       5.000000  10.000000   0.009013  0.009013  0.009013  0.009013  0.009013   
       10.000000 15.000000   0.009013  0.009013  0.009013  0.009013  0.009013   
       15.000000 20.000000   0.008567  0.008567  0.008567  0.008567  0.008567   
       20.000000 25.000000   0.008567  0.008567  0.008567  0.008567  0.008567   
       25.000000 30.000000   0.008567  0.008567  0.008567  0.008567  0.008567   
       30.000000 35.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       35.000000 40.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       40.000000 45.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       45.000000 50.000000   0.008563  0.008563  0.008563  0.008563  0.008563   
       50.000000 55.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       55.000000 60.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       60.000000 65.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       65.000000 70.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       70.000000 75.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       75.000000 80.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       80.000000 85.000000   0.008562  0.008562  0.008562  0.008562  0.008562   
       85.000000 90.000000   0.008562  0.008562  0.008562  0.0

In [30]:
hgb_mean = hgb_mean * scale_factor

In [31]:
assert np.allclose(
    hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum(),
    pre_disparity_groups,
)

In [32]:
counterfactual_hgb_mean = hgb_mean.add(delta_effective_coverage * fortification_hemoglobin_mean_difference, axis=0)
counterfactual_hgb_mean

draw_0      draw_1  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           143.365729  143.514997   
                            highest          145.168195  145.319174   
                            lowest           139.080694  139.225614   
                            middle           141.474880  141.622236   
                            second           141.238458  141.385618   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           102.577358  105.552235   
                            highest          103.772182  106.774749   
                            lowest           100.398621  103.315474   
                            middle           101.782771  104.736810   
                            second           101.379407  104.323790   

                                                 draw_2      draw_3  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           145.408498  144.852923   
                            highest          147.234377  146.672434   
                            lowest           141.063962  140.524570   
                            middle           143.491478  142.943021   
                            second           143.252386  142.704655   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           104.745345  104.606823   
                            highest          105.960348  105.820537   
                            lowest           102.524322  102.388502   
                            middle           103.935572  103.798020   
                            second           103.525170  103.388069   

                                                 draw_4      draw_5  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           143.544929  144.180258   
                            highest          145.349448  145.992059   
                            lowest           139.254674  139.871497   
                            middle           141.651784  142.278973   
                            second           141.415128  142.041486   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           107.105396  105.350549   
                            highest          108.342366  106.571185   
                            lowest           104.838341  103.117722   
                            middle           106.279091  104.536536   
                            second           105.861029  104.124171   

                                                 draw_6      draw_7  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           143.280236  145.251397   
                            highest          145.081722  147.075475   
                            lowest           138.997691  140.911437   
                            middle           141.390483  143.336390   
                            second           141.154172  143.097503   
...                                                 ...         ...   
Male   95.0      125.000000 fourth           105.349775  105.627442   
                            highest          106.570404  106.850655   
                            lowest           103.116963  103.389214   
                            middle           104.535768  104.811489   
                            second           104.123405  104.398225   

                                                 draw_8      draw_9  \
sex    age_start age_end    wealth_quintile                           
Female 0.0       0.019178   fourth           143.730536  145.529608   
                            highest          145.537183  147.356875   
                            lowest           139.434874  141.181545  

In [33]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean").reset_index()

In [34]:
counterfactual_hgb_mean.columns.name = "draw"
counterfactual_hgb_mean = counterfactual_hgb_mean.stack().rename("mean").reset_index()

In [35]:
location_id = utility_data.get_location_id(location.title())
hgb_sd = gbd.get_modelable_entity_draws(me_id=me_ids["hemoglobin_sd"], location_id=location_id, year_id=2021)
hgb_sd = reshape_to_vivarium_format(hgb_sd, location.title()).droplevel(["year_start", "year_end", "measure_id", "metric_id", "model_version_id", "modelable_entity_id"])[DRAWS].copy()
hgb_sd

draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    11.056572  15.419496  12.937057  14.108613   
       0.019178  0.076712    18.143206  17.170122  18.638921  21.675732   
       0.076712  0.500000    11.519491  14.027332  13.723109  14.417308   
       0.500000  1.000000    13.756756  14.711771  13.648304  13.187534   
       1.000000  2.000000    14.325459  16.843282  16.709555  15.864417   
       2.000000  5.000000    16.864334  16.488876  16.574293  16.946859   
       5.000000  10.000000   14.225182  14.168108  12.948419  12.913613   
       10.000000 15.000000   13.709243  13.495770  11.389442  15.925441   
       15.000000 20.000000   13.561928  15.733414  13.751380  15.264811   
       20.000000 25.000000   17.877538  15.796879  16.381351  14.863829   
       25.000000 30.000000   16.815011  16.994822  15.316674  15.690873   
       30.000000 35.000000   16.621743  15.047877  16.867962  15.976469   
       35.000000 40.000000   14.396311  15.728416  16.175030  15.550384   
       40.000000 45.000000   16.718798  15.378787  17.134685  16.801107   
       45.000000 50.000000   17.032453  15.404674  16.578969  18.835821   
       50.000000 55.000000   15.717267  15.389592  14.263427  15.514466   
       55.000000 60.000000   12.958764  14.368135  16.001020  15.681634   
       60.000000 65.000000   14.203046  14.792499  14.244354  14.557884   
       65.000000 70.000000   12.468907  15.525386  14.499738  14.166835   
       70.000000 75.000000   16.333369  16.631346  16.833404  15.235709   
       75.000000 80.000000   22.254054  24.403128  20.115561  23.155481   
       80.000000 85.000000   24.100034  24.539483  24.989374  26.511926   
       85.000000 90.000000   36.306586  33.251947  37.348512  32.257547   
       90.000000 95.000000   38.914197  41.875007  39.460541  40.321736   
       95.000000 125.000000  38.019009  39.773450  39.091306  40.811093   
Male   0.000000  0.019178     7.610899   6.575979   7.453095   6.250333   
       0.019178  0.076712    16.977021  18.465539  16.510252  21.378982   
       0.076712  0.500000     9.721262  12.425339  14.429252  13.617889   
       0.500000  1.000000    14.652405  12.306607  13.147011  14.384103   
       1.000000  2.000000    13.806197  12.868476  13.713812  14.321067   
       2.000000  5.000000    14.572863  13.560881  13.249884  13.447389   
       5.000000  10.000000   12.573757   7.057425   9.593659   9.916708   
       10.000000 15.000000   11.260913  12.769954   9.948129  11.258765   
       15.000000 20.000000   11.470756  15.361375  13.757213  15.839422   
       20.000000 25.000000   16.818841  15.146760  14.406426  15.531786   
       25.000000 30.000000   14.325444  14.905894  14.724530  12.324918   
       30.000000 35.000000   16.155267  16.364407  15.748614  15.395269   
       35.000000 40.000000   17.000588  14.962367  16.688472  15.727331   
       40.000000 45.000000   16.215645  16.148306  15.713011  15.286601   
       45.000000 50.000000   16.866309  17.598236  14.838259  17.183189   
       50.000000 55.000000   16.298999  17.308050  17.941748  18.389619   
       55.000000 60.000000   19.614086  19.237907  18.023727  18.054980   
       60.000000 65.000000   17.017625  19.435720  17.143083  16.953227   
       65.000000 70.000000   17.246434  17.477513  19.252672  18.443892   
       70.000000 75.000000   18.666848  17.255199  17.201503  17.486489   
       75.000000 80.000000   19.081333  16.399954  20.715293  21.595394   
       80.000000 85.000000   21.153456  22.487688  23.106116  22.546762   
       85.000000 90.000000   23.662495  24.946716  26.106795  23.226613   
       90.000000 95.000000   27.527042  30.766501  28.349045  28.340500   
       95.000000 125.000000  39.559564  39.138167  37.453939  35.318036   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

In [36]:
hemoglobin_sd_disparities = pd.read_csv(f'../0100_data_prep/results/hemoglobin/sd_disparities/{location}.csv')
hemoglobin_sd_disparities = (
    map_to_population_age_groups(hemoglobin_sd_disparities[hemoglobin_sd_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_sd_disparities

sex     age_start  age_end     wealth_quintile
Female  0.0        0.019178    lowest             14.697789
                               second             14.743958
                               middle             15.145306
                               fourth             15.145327
                               highest            14.592007
                                                    ...    
Male    95.0       125.000000  lowest             16.135241
                               second             16.281235
                               middle             16.559569
                               fourth             16.266386
                               highest            15.538198
Name: value, Length: 250, dtype: float64

In [37]:
pre_disparity_groups = hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum()
pre_disparity_groups

draw_0     draw_1     draw_2     draw_3  \
sex    age_start age_end                                                  
Female 0.000000  0.019178    11.056572  15.419496  12.937057  14.108613   
       0.019178  0.076712    18.143206  17.170122  18.638921  21.675732   
       0.076712  0.500000    11.519491  14.027332  13.723109  14.417308   
       0.500000  1.000000    13.756756  14.711771  13.648304  13.187534   
       1.000000  2.000000    14.325459  16.843282  16.709555  15.864417   
       2.000000  5.000000    16.864334  16.488876  16.574293  16.946859   
       5.000000  10.000000   14.225182  14.168108  12.948419  12.913613   
       10.000000 15.000000   13.709243  13.495770  11.389442  15.925441   
       15.000000 20.000000   13.561928  15.733414  13.751380  15.264811   
       20.000000 25.000000   17.877538  15.796879  16.381351  14.863829   
       25.000000 30.000000   16.815011  16.994822  15.316674  15.690873   
       30.000000 35.000000   16.621743  15.047877  16.867962  15.976469   
       35.000000 40.000000   14.396311  15.728416  16.175030  15.550384   
       40.000000 45.000000   16.718798  15.378787  17.134685  16.801107   
       45.000000 50.000000   17.032453  15.404674  16.578969  18.835821   
       50.000000 55.000000   15.717267  15.389592  14.263427  15.514466   
       55.000000 60.000000   12.958764  14.368135  16.001020  15.681634   
       60.000000 65.000000   14.203046  14.792499  14.244354  14.557884   
       65.000000 70.000000   12.468907  15.525386  14.499738  14.166835   
       70.000000 75.000000   16.333369  16.631346  16.833404  15.235709   
       75.000000 80.000000   22.254054  24.403128  20.115561  23.155481   
       80.000000 85.000000   24.100034  24.539483  24.989374  26.511926   
       85.000000 90.000000   36.306586  33.251947  37.348512  32.257547   
       90.000000 95.000000   38.914197  41.875007  39.460541  40.321736   
       95.000000 125.000000  38.019009  39.773450  39.091306  40.811093   
Male   0.000000  0.019178     7.610899   6.575979   7.453095   6.250333   
       0.019178  0.076712    16.977021  18.465539  16.510252  21.378982   
       0.076712  0.500000     9.721262  12.425339  14.429252  13.617889   
       0.500000  1.000000    14.652405  12.306607  13.147011  14.384103   
       1.000000  2.000000    13.806197  12.868476  13.713812  14.321067   
       2.000000  5.000000    14.572863  13.560881  13.249884  13.447389   
       5.000000  10.000000   12.573757   7.057425   9.593659   9.916708   
       10.000000 15.000000   11.260913  12.769954   9.948129  11.258765   
       15.000000 20.000000   11.470756  15.361375  13.757213  15.839422   
       20.000000 25.000000   16.818841  15.146760  14.406426  15.531786   
       25.000000 30.000000   14.325444  14.905894  14.724530  12.324918   
       30.000000 35.000000   16.155267  16.364407  15.748614  15.395269   
       35.000000 40.000000   17.000588  14.962367  16.688472  15.727331   
       40.000000 45.000000   16.215645  16.148306  15.713011  15.286601   
       45.000000 50.000000   16.866309  17.598236  14.838259  17.183189   
       50.000000 55.000000   16.298999  17.308050  17.941748  18.389619   
       55.000000 60.000000   19.614086  19.237907  18.023727  18.054980   
       60.000000 65.000000   17.017625  19.435720  17.143083  16.953227   
       65.000000 70.000000   17.246434  17.477513  19.252672  18.443892   
       70.000000 75.000000   18.666848  17.255199  17.201503  17.486489   
       75.000000 80.000000   19.081333  16.399954  20.715293  21.595394   
       80.000000 85.000000   21.153456  22.487688  23.106116  22.546762   
       85.000000 90.000000   23.662495  24.946716  26.106795  23.226613   
       90.000000 95.000000   27.527042  30.766501  28.349045  28.340500   
       95.000000 125.000000  39.559564  39.138167  37.453939  35.318036   

                                draw_4     draw_5     draw_6     draw_7  \
sex    age_start age_end                                

In [38]:
hgb_sd = hgb_sd.mul(hemoglobin_sd_disparities, axis=0)

In [39]:
scale_factor = pre_disparity_groups / hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum()
scale_factor

draw_0    draw_1    draw_2    draw_3    draw_4  \
sex    age_start age_end                                                        
Female 0.000000  0.019178    0.067283  0.067283  0.067283  0.067283  0.067283   
       0.019178  0.076712    0.067283  0.067283  0.067283  0.067283  0.067283   
       0.076712  0.500000    0.067283  0.067283  0.067283  0.067283  0.067283   
       0.500000  1.000000    0.067283  0.067283  0.067283  0.067283  0.067283   
       1.000000  2.000000    0.067283  0.067283  0.067283  0.067283  0.067283   
       2.000000  5.000000    0.067283  0.067283  0.067283  0.067283  0.067283   
       5.000000  10.000000   0.064429  0.064429  0.064429  0.064429  0.064429   
       10.000000 15.000000   0.064429  0.064429  0.064429  0.064429  0.064429   
       15.000000 20.000000   0.061866  0.061866  0.061866  0.061866  0.061866   
       20.000000 25.000000   0.061866  0.061866  0.061866  0.061866  0.061866   
       25.000000 30.000000   0.061866  0.061866  0.061866  0.061866  0.061866   
       30.000000 35.000000   0.061962  0.061962  0.061962  0.061962  0.061962   
       35.000000 40.000000   0.061962  0.061962  0.061962  0.061962  0.061962   
       40.000000 45.000000   0.061962  0.061962  0.061962  0.061962  0.061962   
       45.000000 50.000000   0.061962  0.061962  0.061962  0.061962  0.061962   
       50.000000 55.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       55.000000 60.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       60.000000 65.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       65.000000 70.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       70.000000 75.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       75.000000 80.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       80.000000 85.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       85.000000 90.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       90.000000 95.000000   0.061993  0.061993  0.061993  0.061993  0.061993   
       95.000000 125.000000  0.061993  0.061993  0.061993  0.061993  0.061993   
Male   0.000000  0.019178    0.066264  0.066264  0.066264  0.066264  0.066264   
       0.019178  0.076712    0.066264  0.066264  0.066264  0.066264  0.066264   
       0.076712  0.500000    0.066264  0.066264  0.066264  0.066264  0.066264   
       0.500000  1.000000    0.066264  0.066264  0.066264  0.066264  0.066264   
       1.000000  2.000000    0.066264  0.066264  0.066264  0.066264  0.066264   
       2.000000  5.000000    0.066264  0.066264  0.066264  0.066264  0.066264   
       5.000000  10.000000   0.063976  0.063976  0.063976  0.063976  0.063976   
       10.000000 15.000000   0.063976  0.063976  0.063976  0.063976  0.063976   
       15.000000 20.000000   0.061872  0.061872  0.061872  0.061872  0.061872   
       20.000000 25.000000   0.061872  0.061872  0.061872  0.061872  0.061872   
       25.000000 30.000000   0.061872  0.061872  0.061872  0.061872  0.061872   
       30.000000 35.000000   0.061947  0.061947  0.061947  0.061947  0.061947   
       35.000000 40.000000   0.061947  0.061947  0.061947  0.061947  0.061947   
       40.000000 45.000000   0.061947  0.061947  0.061947  0.061947  0.061947   
       45.000000 50.000000   0.061947  0.061947  0.061947  0.061947  0.061947   
       50.000000 55.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       55.000000 60.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       60.000000 65.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       65.000000 70.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       70.000000 75.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       75.000000 80.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       80.000000 85.000000   0.062029  0.062029  0.062029  0.062029  0.062029   
       85.000000 90.000000   0.062029  0.062029  0.062029  0.0

In [40]:
hgb_sd = hgb_sd * scale_factor

In [41]:
assert np.allclose(
    hgb_sd.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_sd.index.names if c != 'wealth_quintile']).sum(),
    pre_disparity_groups,
)

In [42]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd").reset_index()
hgb_sd

,sex,age_start,age_end,wealth_quintile,draw,sd
0,Female,0.0,0.019178,lowest,draw_0,10.934018
1,Female,0.0,0.019178,lowest,draw_1,15.248582
2,Female,0.0,0.019178,lowest,draw_2,12.793660
3,Female,0.0,0.019178,lowest,draw_3,13.952230
4,Female,0.0,0.019178,lowest,draw_4,12.875777
...,...,...,...,...,...,...
124995,Male,95.0,125.000000,highest,draw_495,38.237787
124996,Male,95.0,125.000000,highest,draw_496,36.635558
124997,Male,95.0,125.000000,highest,draw_497,36.002955
124998,Male,95.0,125.000000,highest,draw_498,34.653317


In [43]:
mean_and_sd_hgb = pd.concat([
    hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='baseline'),
    counterfactual_hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='intervention')
])
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario
0,Female,0.0,0.019178,lowest,draw_0,138.477305,10.934018,baseline
1,Female,0.0,0.019178,lowest,draw_1,138.622225,15.248582,baseline
2,Female,0.0,0.019178,lowest,draw_2,140.460574,12.793660,baseline
3,Female,0.0,0.019178,lowest,draw_3,139.921181,13.952230,baseline
4,Female,0.0,0.019178,lowest,draw_4,138.651285,12.875777,baseline
...,...,...,...,...,...,...,...,...
124995,Male,95.0,125.000000,second,draw_495,102.264838,40.066317,intervention
124996,Male,95.0,125.000000,second,draw_496,102.548445,38.387469,intervention
124997,Male,95.0,125.000000,second,draw_497,101.871760,37.724615,intervention
124998,Male,95.0,125.000000,second,draw_498,103.187056,36.310437,intervention


In [44]:
thresholds = reshape_to_vivarium_format(pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv'), location.title()).droplevel(["age_group_name", "grp"]).reset_index()
thresholds

,sex,age_start,age_end,hgb_lower_anemic,hgb_lower_mild,hgb_lower_moderate,hgb_lower_severe,hgb_upper_anemic,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe,pregnant
0,Female,0.000000,0.019178,0,145,100,0,160,160,145,100,0
1,Female,0.019178,0.076712,0,120,85,0,135,135,120,85,0
2,Female,0.076712,0.500000,0,100,70,0,110,110,100,70,0
3,Female,0.500000,1.000000,0,100,70,0,110,110,100,70,0
4,Female,1.000000,2.000000,0,100,70,0,110,110,100,70,0
5,Female,2.000000,5.000000,0,100,70,0,110,110,100,70,0
6,Female,5.000000,10.000000,0,110,80,0,115,115,110,80,0
7,Female,10.000000,15.000000,0,100,70,0,110,110,100,70,1
8,Female,10.000000,15.000000,0,110,80,0,115,115,110,80,0
9,Female,15.000000,20.000000,0,110,80,0,120,120,110,80,0


In [45]:
mean_and_sd_hgb = mean_and_sd_hgb.assign(pregnant=0).merge(thresholds, on=["sex", "age_start", "age_end", "pregnant"], how="left", validate="m:1")
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario,pregnant,hgb_lower_anemic,hgb_lower_mild,hgb_lower_moderate,hgb_lower_severe,hgb_upper_anemic,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe
0,Female,0.0,0.019178,lowest,draw_0,138.477305,10.934018,baseline,0,0,145,100,0,160,160,145,100
1,Female,0.0,0.019178,lowest,draw_1,138.622225,15.248582,baseline,0,0,145,100,0,160,160,145,100
2,Female,0.0,0.019178,lowest,draw_2,140.460574,12.793660,baseline,0,0,145,100,0,160,160,145,100
3,Female,0.0,0.019178,lowest,draw_3,139.921181,13.952230,baseline,0,0,145,100,0,160,160,145,100
4,Female,0.0,0.019178,lowest,draw_4,138.651285,12.875777,baseline,0,0,145,100,0,160,160,145,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,Male,95.0,125.000000,second,draw_495,102.264838,40.066317,intervention,0,0,110,80,0,130,130,110,80
249996,Male,95.0,125.000000,second,draw_496,102.548445,38.387469,intervention,0,0,110,80,0,130,130,110,80
249997,Male,95.0,125.000000,second,draw_497,101.871760,37.724615,intervention,0,0,110,80,0,130,130,110,80
249998,Male,95.0,125.000000,second,draw_498,103.187056,36.310437,intervention,0,0,110,80,0,130,130,110,80


In [46]:
assert (
    (mean_and_sd_hgb.hgb_upper_mild == mean_and_sd_hgb.hgb_upper_anemic).all() &
    (mean_and_sd_hgb.hgb_lower_severe == mean_and_sd_hgb.hgb_lower_anemic).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [47]:
assert (
    (mean_and_sd_hgb.hgb_lower_mild == mean_and_sd_hgb.hgb_upper_moderate).all() &
    (mean_and_sd_hgb.hgb_lower_moderate == mean_and_sd_hgb.hgb_upper_severe).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [48]:
def _hemoglobin_distribution_parts_from_mean_sd(mean, sd):
    # NOTE: This is an unusual ensemble distribution. We should add functionality to the
    # EnsembleDistribution class to make this easier.
    x_min = 0
    x_max = 220
    gamma_params = risk_distributions.risk_distributions.Gamma.get_parameters(
        mean=mean, sd=sd
    )
    # NOTE: We have to override these, otherwise Gamma is overly conservative in what values
    # are computable
    # https://github.com/ihmeuw/risk_distributions/issues/61
    gamma_params["x_min"] = x_min
    gamma_params["x_max"] = x_max
    hemoglobin_distribution_gamma_part = risk_distributions.risk_distributions.Gamma(
        gamma_params
    )

    # NOTE: Forced to duplicate https://github.com/ihmeuw/risk_distributions/blob/a9ed9d7e8372590018355012a7a7ffefa87b0819/src/risk_distributions/risk_distributions.py#L428-L434
    # because it doesn't permit the custom x_min and x_max, and these are used in calculating the others
    mgumbel_params = pd.DataFrame({
        "loc": x_max - mean - (np.euler_gamma * np.sqrt(6) / np.pi * sd),
        "scale": np.sqrt(6) / np.pi * sd,
        "x_min": x_min,
        "x_max": x_max,
    })
    hemoglobin_distribution_mgumbel_part = (
        risk_distributions.risk_distributions.MirroredGumbel(mgumbel_params)
    )
    return hemoglobin_distribution_gamma_part, hemoglobin_distribution_mgumbel_part

(
    hemoglobin_distribution_gamma_part,
    hemoglobin_distribution_mgumbel_part,
) = _hemoglobin_distribution_parts_from_mean_sd(mean_and_sd_hgb['mean'], mean_and_sd_hgb.sd)

def cdf(x):
    gamma_cdf = hemoglobin_distribution_gamma_part.cdf(x)
    # NOTE: There is a bug in this CDF function -- it is reversed!
    # https://github.com/ihmeuw/risk_distributions/issues/62
    mgumbel_cdf = 1 - hemoglobin_distribution_mgumbel_part.cdf(x)
    return (
        0.4
        * gamma_cdf
        + 0.6
        * mgumbel_cdf
    )

In [49]:
mean_and_sd_hgb["severe"] = cdf(mean_and_sd_hgb.hgb_upper_severe.copy()) - cdf(mean_and_sd_hgb.hgb_lower_severe.copy())
mean_and_sd_hgb["moderate"] = cdf(mean_and_sd_hgb.hgb_upper_moderate.copy()) - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["mild"] = cdf(mean_and_sd_hgb.hgb_upper_mild.copy()) - mean_and_sd_hgb["moderate"].copy() - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["anemic"] = mean_and_sd_hgb["mild"] + mean_and_sd_hgb["moderate"] + mean_and_sd_hgb["severe"]
mean_and_sd_hgb

,sex,age_start,age_end,wealth_quintile,draw,mean,sd,scenario,pregnant,hgb_lower_severe,hgb_upper_mild,hgb_upper_moderate,hgb_upper_severe,severe,moderate,mild,anemic
0,Female,0.0,0.019178,lowest,draw_0,138.477305,10.934018,baseline,0,0,160,145,100,0.003702,0.708900,0.275391,0.987992
1,Female,0.0,0.019178,lowest,draw_1,138.622225,15.248582,baseline,0,0,160,145,100,0.014031,0.625490,0.306242,0.945762
2,Female,0.0,0.019178,lowest,draw_2,140.460574,12.793660,baseline,0,0,160,145,100,0.005898,0.605890,0.349860,0.961648
3,Female,0.0,0.019178,lowest,draw_3,139.921181,13.952230,baseline,0,0,160,145,100,0.008843,0.607235,0.334980,0.951058
4,Female,0.0,0.019178,lowest,draw_4,138.651285,12.875777,baseline,0,0,160,145,100,0.007315,0.663041,0.302771,0.973128
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,Male,95.0,125.000000,second,draw_495,102.264838,40.066317,intervention,0,0,130,110,80,0.259645,0.298072,0.200246,0.757964
249996,Male,95.0,125.000000,second,draw_496,102.548445,38.387469,intervention,0,0,130,110,80,0.250609,0.306695,0.208659,0.765963
249997,Male,95.0,125.000000,second,draw_497,101.871760,37.724615,intervention,0,0,130,110,80,0.253340,0.312616,0.210960,0.776916
249998,Male,95.0,125.000000,second,draw_498,103.187056,36.310437,intervention,0,0,130,110,80,0.236032,0.317487,0.220419,0.773938


In [50]:
disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
disability_weights.columns.name = 'draw'
disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
disability_weights

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


In [51]:
mean_and_sd_hgb = mean_and_sd_hgb.merge(
    disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
    validate="m:1",
)

In [52]:
mean_and_sd_hgb["mild_ylds"] = mean_and_sd_hgb.mild * mean_and_sd_hgb.mild_dw
mean_and_sd_hgb["moderate_ylds"] = mean_and_sd_hgb.moderate * mean_and_sd_hgb.moderate_dw
mean_and_sd_hgb["severe_ylds"] = mean_and_sd_hgb.severe * mean_and_sd_hgb.severe_dw
mean_and_sd_hgb['anemic_ylds'] = mean_and_sd_hgb['mild_ylds'] + mean_and_sd_hgb['moderate_ylds'] + mean_and_sd_hgb['severe_ylds']

In [53]:
index_cols = ['age_start', 'age_end', 'sex', 'draw', "wealth_quintile"]
value_cols = ["mild", "moderate", "severe", "anemic", "mild_ylds", "moderate_ylds", "severe_ylds", "anemic_ylds"]

baseline_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'baseline'].set_index(index_cols)[value_cols]
counterfactual_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'intervention'].set_index(index_cols)[value_cols]

In [54]:
baseline_anemia.sort_index()

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.413781  0.546932   
                                    highest          0.463321  0.487676   
                                    lowest           0.275391  0.708900   
                                    middle           0.357673  0.615561   
                                    second           0.349879  0.627360   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.214769  0.304197   
                                    highest          0.225362  0.310701   
                                    lowest           0.213739  0.312407   
                                    middle           0.210380  0.302651   
                                    second           0.213321  0.307401   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.002629  0.963342   
                                    highest          0.001802  0.952800   
                                    lowest           0.003702  0.987992   
                                    middle           0.003241  0.976474   
                                    second           0.002917  0.980155   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.230727  0.749693   
                                    highest          0.214819  0.750882   
                                    lowest           0.245789  0.771936   
                                    middle           0.239584  0.752615   
                                    second           0.239426  0.760147   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.001002       0.032433   
                                    highest           0.001121       0.028919   
                                    lowest            0.000667       0.042038   
                                    middle            0.000866       0.036503   
                                    second            0.000847       0.037203   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.000902       0.012679   
                                    highest           0.000947       0.012950   
                                    lowest            0.000898       0.013021   
                                    middle            0.000884       0.012615   
                                    second            0.000896       0.012813   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.000523     0.033958  
                                    highest             0.000358     0.030399  
                                    lowest              0.000736     0.043441  
                                    middle              0.000645     0.038013  
                                    second              0.000580     0.038630  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.027407     0.040988  
                                    highest             0.025517     0.039414  
                                    lowest              0.029196     0.043115  
                                    middle              0.028459     0.041957  
                                    second              0.028440     0.042149  

[125000 ro

In [55]:
counterfactual_anemia.sort_index()

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.434127  0.519577   
                                    highest          0.485402  0.453858   
                                    lowest           0.296323  0.686518   
                                    middle           0.378990  0.590375   
                                    second           0.370811  0.603425   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.215401  0.302192   
                                    highest          0.226245  0.307459   
                                    lowest           0.214354  0.311121   
                                    middle           0.210957  0.301026   
                                    second           0.213885  0.305968   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.002417  0.956121   
                                    highest          0.001620  0.940880   
                                    lowest           0.003446  0.986287   
                                    middle           0.003001  0.972366   
                                    second           0.002712  0.976948   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.225941  0.743534   
                                    highest          0.208121  0.741826   
                                    lowest           0.242340  0.767814   
                                    middle           0.235406  0.747389   
                                    second           0.235758  0.755611   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.001051       0.030811   
                                    highest           0.001175       0.026914   
                                    lowest            0.000717       0.040711   
                                    middle            0.000917       0.035010   
                                    second            0.000898       0.035783   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.000905       0.012596   
                                    highest           0.000950       0.012815   
                                    lowest            0.000900       0.012968   
                                    middle            0.000886       0.012547   
                                    second            0.000898       0.012753   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.000481     0.032343  
                                    highest             0.000322     0.028411  
                                    lowest              0.000686     0.042114  
                                    middle              0.000597     0.036524  
                                    second              0.000539     0.037220  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.026839     0.040339  
                                    highest             0.024722     0.038487  
                                    lowest              0.028786     0.042655  
                                    middle              0.027963     0.041396  
                                    second              0.028005     0.041656  

[125000 ro

In [56]:
baseline_anemia - counterfactual_anemia

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth          -0.020346  0.027355   
                                    highest         -0.022081  0.033818   
                                    lowest          -0.020932  0.022382   
                                    middle          -0.021317  0.025185   
                                    second          -0.020933  0.023935   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth          -0.000631  0.002005   
                                    highest         -0.000883  0.003242   
                                    lowest          -0.000614  0.001287   
                                    middle          -0.000577  0.001625   
                                    second          -0.000564  0.001433   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.000211  0.007221   
                                    highest          0.000182  0.011920   
                                    lowest           0.000256  0.001706   
                                    middle           0.000240  0.004108   
                                    second           0.000205  0.003207   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.004785  0.006159   
                                    highest          0.006697  0.009056   
                                    lowest           0.003449  0.004121   
                                    middle           0.004178  0.005226   
                                    second           0.003667  0.004536   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth           -0.000049       0.001622   
                                    highest          -0.000053       0.002005   
                                    lowest           -0.000051       0.001327   
                                    middle           -0.000052       0.001494   
                                    second           -0.000051       0.001419   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth           -0.000003       0.000084   
                                    highest          -0.000004       0.000135   
                                    lowest           -0.000003       0.000054   
                                    middle           -0.000002       0.000068   
                                    second           -0.000002       0.000060   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.000042     0.001615  
                                    highest             0.000036     0.001988  
                                    lowest              0.000051     0.001327  
                                    middle              0.000048     0.001490  
                                    second              0.000041     0.001409  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.000568     0.000649  
                                    highest             0.000796     0.000927  
                                    lowest              0.000410     0.000461  
                                    middle              0.000496     0.000562  
                                    second              0.000436     0.000493  

[125000 ro

In [57]:
# BUT our counterfactual is only in a world where everyone is iron-responsive.
# Cleaned this up from https://github.com/ihmeuw/vivarium_research_lsff/blob/1cb465a752d299401ae366db537dc8d557162184/multiplication_models/iron_model_U5.ipynb,
# but have not checked it in extreme detail.
iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.moderate_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.severe_anemia_due_to_schistosomiasis,
    gbd_mapping.sequelae.mild_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hookworm_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_neglected_tropical_diseases,
    gbd_mapping.sequelae.mild_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.moderate_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.severe_anemia_due_to_maternal_hemorrhage,
    gbd_mapping.sequelae.mild_iron_deficiency_anemia,
    gbd_mapping.sequelae.moderate_iron_deficiency_anemia,
    gbd_mapping.sequelae.severe_iron_deficiency_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.severe_anemia_due_to_other_infectious_diseases,
    gbd_mapping.sequelae.menstrual_disorders_with_mild_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_moderate_anemia,
    gbd_mapping.sequelae.menstrual_disorders_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_mild_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_moderate_anemia,
    gbd_mapping.sequelae.uterine_fibroids_symptomatic_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_pud_with_severe_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.mildly_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.mildy_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.moderately_symptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.asymptomatic_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_3_chronic_kidney_disease_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mdoerate_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_4_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.stage_5_chronic_kidney_disease_untreated_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.vitamin_a_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_mild_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_moderate_anemia,
    gbd_mapping.sequelae.ulcerative_colitis_with_severe_anemia,
    gbd_mapping.sequelae.crohns_disease_with_mild_anemia,
    gbd_mapping.sequelae.crohns_disease_with_moderate_anemia,
    gbd_mapping.sequelae.crohns_disease_with_severe_anemia,
    gbd_mapping.sequelae.complicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.complicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.complicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_pud_with_severe_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_mild_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_moderate_anemia,
    gbd_mapping.sequelae.severe_acute_uncomplicated_gastritis_duodenitis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_mild_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_moderate_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_1_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_type_2_diabetes_mellitus_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_hypertension_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_glomerulonephritis_with_severe_anemia,
    gbd_mapping.sequelae.end_stage_renal_disease_on_dialysis_due_to_other_and_unspecified_causes_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_b_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_hepatitis_c_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_alcohol_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_other_decompensated_with_severe_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_mild_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_moderate_anemia,
    gbd_mapping.sequelae.cirrhosis_and_other_chronic_liver_diseases_due_to_nash_decompensated_with_severe_anemia,
]

In [58]:
non_iron_responsive_anemia_sequelae = [
    gbd_mapping.sequelae.mild_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.moderate_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.severe_anemia_due_to_other_hemoglobinopathies_and_hemolytic_anemias,
    gbd_mapping.sequelae.mild_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_b_thalassemia_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_e_trait,
    gbd_mapping.sequelae.mild_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.moderate_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.severe_anemia_due_to_sickle_cell_trait,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_mild_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_moderate_anemia,
    gbd_mapping.sequelae.hemizygous_g6pd_deficiency_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_parasitemia_pfpr,
    gbd_mapping.sequelae.mild_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_homozygous_sickle_cell_and_severe_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.mild_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.moderate_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.severe_anemia_due_to_hemoglobin_sc_disease,
    gbd_mapping.sequelae.mild_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.moderate_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_anemia_due_to_mild_sickle_cell_beta_thalassemia,
    gbd_mapping.sequelae.severe_malaria_with_mild_anemia,
    gbd_mapping.sequelae.severe_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.severe_malaria_with_severe_anemia,
    gbd_mapping.sequelae.mild_malaria_with_mild_anemia,
    gbd_mapping.sequelae.mild_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.mild_malaria_with_severe_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_mild_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_moderate_anemia,
    gbd_mapping.sequelae.moderate_malaria_with_severe_anemia,
    gbd_mapping.sequelae.early_hiv_with_mild_anemia,
    gbd_mapping.sequelae.early_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.early_hiv_with_severe_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_mild_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_moderate_anemia,
    gbd_mapping.sequelae.symptomatic_hiv_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_with_antiretroviral_treatment_with_severe_anemia,
    gbd_mapping.sequelae.aids_with_mild_anemia,
    gbd_mapping.sequelae.aids_with_moderate_anemia,
    gbd_mapping.sequelae.aids_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_drug_susceptible_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_multidrug_resistant_tuberculosis_without_extensive_drug_resistance_with_severe_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_mild_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_moderate_anemia,
    gbd_mapping.sequelae.hiv_aids_extensively_drug_resistant_tuberculosis_with_severe_anemia,
    gbd_mapping.sequelae.mild_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.moderate_anemia_due_to_malaria_vivax_pvpr,
    gbd_mapping.sequelae.severe_anemia_due_to_malaria_vivax_pvpr,
]

In [59]:
len(iron_responsive_anemia_sequelae)

138

In [60]:
len(non_iron_responsive_anemia_sequelae)

60

In [61]:
loguru.logger.disable("vivarium_inputs.validation.raw")

In [62]:
iron_responsive_prevalence = None

for sequela in tqdm(iron_responsive_anemia_sequelae):
    try:
        sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location", "year_start", "year_end"])
    except DataDoesNotExistError as e:
        assert 'zero' in str(e)
        continue
    except DataAbnormalError as e:
        assert 'zero' in str(e)
        continue

    if iron_responsive_prevalence is None:
        iron_responsive_prevalence = sequela_prevalence
    else:
        iron_responsive_prevalence += sequela_prevalence

  0%|          | 0/138 [00:00<?, ?it/s]

In [63]:
non_iron_responsive_prevalence = None

for sequela in tqdm(non_iron_responsive_anemia_sequelae):
    try:
        sequela_prevalence = vivarium_inputs.get_measure(sequela, "prevalence", location.title()).droplevel(["location", "year_start", "year_end"])
    except DataDoesNotExistError as e:
        assert 'zero' in str(e)
        continue
    except DataAbnormalError as e:
        assert 'zero' in str(e)
        continue

    if non_iron_responsive_prevalence is None:
        non_iron_responsive_prevalence = sequela_prevalence
    else:
        non_iron_responsive_prevalence += sequela_prevalence

  0%|          | 0/60 [00:00<?, ?it/s]

In [64]:
loguru.logger.enable("vivarium_inputs.validation.raw")

In [65]:
iron_responsive_prevalence.columns.name = "draw"
iron_responsive_prevalence = iron_responsive_prevalence.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
iron_responsive_prevalence

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.870859
                               draw_1      0.834114
                               draw_2      0.837499
                               draw_3      0.801072
                               draw_4      0.794379
                                             ...   
Male    95.0       125.000000  draw_495    0.657506
                               draw_496    0.624656
                               draw_497    0.643068
                               draw_498    0.633495
                               draw_499    0.653186
Length: 25000, dtype: float64

In [66]:
non_iron_responsive_prevalence.columns.name = "draw"
non_iron_responsive_prevalence = non_iron_responsive_prevalence.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
non_iron_responsive_prevalence

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.056061
                               draw_1      0.052256
                               draw_2      0.055149
                               draw_3      0.047436
                               draw_4      0.053121
                                             ...   
Male    95.0       125.000000  draw_495    0.097808
                               draw_496    0.091776
                               draw_497    0.093861
                               draw_498    0.093842
                               draw_499    0.097005
Length: 25000, dtype: float64

In [67]:
iron_responsive_proportion = iron_responsive_prevalence / (iron_responsive_prevalence + non_iron_responsive_prevalence)
iron_responsive_proportion

sex     age_start  age_end     draw    
Female  0.0        0.019178    draw_0      0.939519
                               draw_1      0.941045
                               draw_2      0.938219
                               draw_3      0.944095
                               draw_4      0.937320
                                             ...   
Male    95.0       125.000000  draw_495    0.870507
                               draw_496    0.871899
                               draw_497    0.872632
                               draw_498    0.870979
                               draw_499    0.870693
Length: 25000, dtype: float64

In [68]:
assert (iron_responsive_proportion <= 1).all()

In [69]:
iron_responsive_proportion.sort_values()

sex     age_start  age_end    draw    
Male    15.0       20.000000  draw_329    0.726722
                              draw_338    0.730490
        20.0       25.000000  draw_314    0.737482
        15.0       20.000000  draw_4      0.737639
        20.0       25.000000  draw_123    0.738444
                                            ...   
Female  0.0        0.019178   draw_60     0.947937
                              draw_83     0.948082
                              draw_220    0.948098
                              draw_425    0.948795
                              draw_75     0.948839
Length: 25000, dtype: float64

In [70]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part, since they all depend on CDFs which combine
# this way.

counterfactual_anemia_accounting_for_non_response = (
    counterfactual_anemia.mul(iron_responsive_proportion, axis=0) +
    baseline_anemia.mul(1 - iron_responsive_proportion, axis=0)
)
counterfactual_anemia_accounting_for_non_response

mild  moderate  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.432896  0.521232   
                                    highest          0.484067  0.455904   
                                    lowest           0.295057  0.687872   
                                    middle           0.377700  0.591898   
                                    second           0.369545  0.604873   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.215312  0.302474   
                                    highest          0.226121  0.307916   
                                    lowest           0.214267  0.311302   
                                    middle           0.210876  0.301255   
                                    second           0.213806  0.306169   

                                                       severe    anemic  \
age_start age_end    sex    draw    wealth_quintile                       
0.0       0.019178   Female draw_0  fourth           0.002430  0.956558   
                                    highest          0.001631  0.941601   
                                    lowest           0.003462  0.986390   
                                    middle           0.003015  0.972614   
                                    second           0.002724  0.977142   
...                                                       ...       ...   
95.0      125.000000 Male   draw_99 fourth           0.226615  0.744401   
                                    highest          0.209064  0.743101   
                                    lowest           0.242825  0.768394   
                                    middle           0.235994  0.748124   
                                    second           0.236275  0.756250   

                                                     mild_ylds  moderate_ylds  \
age_start age_end    sex    draw    wealth_quintile                             
0.0       0.019178   Female draw_0  fourth            0.001048       0.030909   
                                    highest           0.001172       0.027035   
                                    lowest            0.000714       0.040791   
                                    middle            0.000914       0.035100   
                                    second            0.000894       0.035869   
...                                                        ...            ...   
95.0      125.000000 Male   draw_99 fourth            0.000904       0.012607   
                                    highest           0.000950       0.012834   
                                    lowest            0.000900       0.012975   
                                    middle            0.000886       0.012557   
                                    second            0.000898       0.012761   

                                                     severe_ylds  anemic_ylds  
age_start age_end    sex    draw    wealth_quintile                            
0.0       0.019178   Female draw_0  fourth              0.000483     0.032441  
                                    highest             0.000324     0.028531  
                                    lowest              0.000689     0.042194  
                                    middle              0.000600     0.036614  
                                    second              0.000542     0.037306  
...                                                          ...          ...  
95.0      125.000000 Male   draw_99 fourth              0.026919     0.040430  
                                    highest             0.024834     0.038618  
                                    lowest              0.028844     0.042719  
                                    middle              0.028033     0.041475  
                                    second              0.028066     0.041725  

[125000 ro

In [71]:
assert ((baseline_anemia - counterfactual_anemia_accounting_for_non_response).anemic_ylds > 0).all()

In [72]:
baseline_ylds = (baseline_anemia.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)
baseline_ylds

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             1059.997610
                               highest             769.541848
                               lowest             1801.271863
                               middle             1244.355229
                               second             1419.783311
                                                     ...     
95.0       125.000000  Male    fourth              894.100061
                               highest            1059.690219
                               lowest              827.604317
                               middle              868.092670
                               second              844.438359
Length: 250, dtype: float64

In [73]:
intervention_ylds = (counterfactual_anemia_accounting_for_non_response.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)

In [74]:
baseline_ylds.groupby(["wealth_quintile"]).sum() - intervention_ylds.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     134418.148177
highest    168446.428250
lowest     123170.634922
middle     128389.677916
second     117541.167482
dtype: float64

In [75]:
ylds = pd.concat([
    baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
    intervention_ylds.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,1059.997610,baseline
1,0.0,0.019178,Female,highest,769.541848,baseline
2,0.0,0.019178,Female,lowest,1801.271863,baseline
3,0.0,0.019178,Female,middle,1244.355229,baseline
4,0.0,0.019178,Female,second,1419.783311,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,881.484946,intervention
496,95.0,125.000000,Male,highest,1037.502802,intervention
497,95.0,125.000000,Male,lowest,819.728976,intervention
498,95.0,125.000000,Male,middle,857.744221,intervention


In [76]:
results_dir = f'./results/{vehicle.lower()}/{location.lower()}/{intervention_scenario.lower()}'

In [77]:
path = f'{results_dir}/ylds.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylds.to_parquet(path)

In [78]:
baseline_anemia_prevalence = baseline_anemia['anemic'].unstack("draw").mean(axis=1)
baseline_anemia_prevalence

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             0.917220
                               highest            0.900539
                               lowest             0.965487
                               middle             0.940266
                               second             0.947353
                                                    ...   
95.0       125.000000  Male    fourth             0.762443
                               highest            0.763989
                               lowest             0.784717
                               middle             0.765232
                               second             0.772893
Length: 250, dtype: float64

In [79]:
baseline_anemia_cases = baseline_anemia_prevalence.mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             33424.279506
                               highest            26298.067393
                               lowest             47936.142149
                               middle             36378.268813
                               second             41281.501854
                                                      ...     
95.0       125.000000  Male    fourth             13163.809909
                               highest            16264.929961
                               lowest             11909.410763
                               middle             12526.984211
                               second             12249.198027
Length: 250, dtype: float64

In [80]:
baseline_anemia_cases.sum() / non_pregnant_pop.sum()

0.4130474930130347

In [81]:
baseline_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.392112
highest    0.367032
lowest     0.464735
middle     0.416044
second     0.426803
dtype: float64

In [82]:
intervention_anemia_prevalence = counterfactual_anemia_accounting_for_non_response['anemic'].unstack("draw").mean(axis=1)
intervention_anemia_prevalence

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             0.906810
                               highest            0.885088
                               lowest             0.961508
                               middle             0.933052
                               second             0.941240
                                                    ...   
95.0       125.000000  Male    fourth             0.757056
                               highest            0.756073
                               lowest             0.781134
                               middle             0.760664
                               second             0.768936
Length: 250, dtype: float64

In [83]:
anemia_prevalence = pd.concat([
    baseline_anemia_prevalence.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_prevalence.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_prevalence

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,0.917220,baseline
1,0.0,0.019178,Female,highest,0.900539,baseline
2,0.0,0.019178,Female,lowest,0.965487,baseline
3,0.0,0.019178,Female,middle,0.940266,baseline
4,0.0,0.019178,Female,second,0.947353,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,0.757056,intervention
496,95.0,125.000000,Male,highest,0.756073,intervention
497,95.0,125.000000,Male,lowest,0.781134,intervention
498,95.0,125.000000,Male,middle,0.760664,intervention


In [84]:
path = f'{results_dir}/anemia_prevalence.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_prevalence.to_parquet(path)

In [85]:
intervention_anemia_cases = intervention_anemia_prevalence.mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

age_start  age_end     sex     wealth_quintile
0.0        0.019178    Female  fourth             33044.907875
                               highest            25846.870322
                               lowest             47738.591671
                               middle             36099.172689
                               second             41015.107357
                                                      ...     
95.0       125.000000  Male    fourth             13070.805034
                               highest            16096.398796
                               lowest             11855.030768
                               middle             12452.202185
                               second             12186.489671
Length: 250, dtype: float64

In [86]:
intervention_anemia_cases.sum() / non_pregnant_pop.sum()

0.4008263788165891

In [87]:
intervention_anemia_cases.groupby(["wealth_quintile"]).sum() / non_pregnant_pop.groupby(["wealth_quintile"]).sum()

wealth_quintile
fourth     0.379731
highest    0.350495
lowest     0.454383
middle     0.404654
second     0.416440
dtype: float64

In [88]:
(baseline_anemia_cases.groupby(["wealth_quintile"]).sum() - intervention_anemia_cases.groupby(["wealth_quintile"]).sum()).map(lambda x: f'{round(x):,.0f}')

wealth_quintile
fourth     3,472,008
highest    4,657,447
lowest     2,839,037
middle     3,181,152
second     2,876,375
dtype: object

In [89]:
anemia_cases = pd.concat([
    baseline_anemia_cases.rename("value").reset_index().assign(scenario="baseline"),
    intervention_anemia_cases.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
anemia_cases

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,33424.279506,baseline
1,0.0,0.019178,Female,highest,26298.067393,baseline
2,0.0,0.019178,Female,lowest,47936.142149,baseline
3,0.0,0.019178,Female,middle,36378.268813,baseline
4,0.0,0.019178,Female,second,41281.501854,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,13070.805034,intervention
496,95.0,125.000000,Male,highest,16096.398796,intervention
497,95.0,125.000000,Male,lowest,11855.030768,intervention
498,95.0,125.000000,Male,middle,12452.202185,intervention


In [90]:
path = f'{results_dir}/anemia_cases.parquet'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
anemia_cases.to_parquet(path)